# Approximating a sine function with PyTorch

We will learn the function

$$f(x) = \sin(2\pi x - 0.2), \qquad 0 < x < 1,$$

from examples using **supervised learning**. Each training example is a pair: an input $x$ and its known target $y=f(x)$.

In [ ]:
# Uncomment the next line if these packages are missing from your notebook kernel.
# %pip install torch matplotlib

import torch
from torch import nn
import matplotlib.pyplot as plt

torch.manual_seed(0)  # Make the initial network weights reproducible.


## 1. Generate labeled training data

In [ ]:
def true_function(x):
    return torch.sin(2 * torch.pi * x - 0.2)

In [ ]:
n_train = 25
x_train = torch.linspace(0, 1, n_train + 2)[1:-1].reshape(-1, 1)
y_train = true_function(x_train)

print("Input shape:", x_train.shape)
print("Target shape:", y_train.shape)

plt.figure(figsize=(7, 3))
plt.scatter(x_train.numpy(), y_train.numpy(), label="Training examples")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Supervised training data")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 2. Define the model

In [ ]:
########################################################################
# Linear regression model with one input feature and one output value.

model = nn.Linear(1, 1)

########################################################################
# Quadratic regression model with one input feature and one output value.

# class QuadraticModel(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.linear = nn.Linear(2, 1)  # Weights for [x, x^2] plus a bias term.

#     def forward(self, x):
#         features = torch.cat([x, x**2], dim=1)  # Shape (N, 1) -> (N, 2).
#         return self.linear(features)
# model = QuadraticModel()

########################################################################
# RBF model: Gaussian features at fixed centers, followed by a linear layer.

# class RBFModel(nn.Module):
#     def __init__(self, n_centers=10, sigma=0.1):
#         super().__init__()
#         self.centers = torch.linspace(0, 1, n_centers).reshape(1, -1)  # Shape (1, n_centers).
#         self.sigma = sigma
#         self.linear = nn.Linear(n_centers, 1)  # One weight per basis function, plus a bias.

#     def forward(self, x):
#         features = torch.exp(-(x - self.centers) ** 2 / (2 * self.sigma**2))  # (N, 1) -> (N, n_centers).
#         return self.linear(features)
# model = RBFModel(n_centers=5, sigma=0.2)

########################################################################
# Neural network with one hidden layer of 8 neurons and a Tanh activation function.

# model = nn.Sequential(
#     nn.Linear(1, 8),  # One input feature -> 8 hidden neurons.
#     nn.Tanh(),
#     nn.Linear(8, 1),  # Hidden neurons -> one predicted value.
# )

########################################################################
print(model)


## 3. Train with mean squared error

We minimize the **mean squared error (MSE)** between predictions and targets:

$$L(\theta)=\frac{1}{N}\sum_{i=1}^{N}\big(\hat y_i-y_i\big)^2.$$

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
n_epochs = 3000
loss_history = []

model.train()
for epoch in range(n_epochs):
    y_pred = model(x_train)           # Forward pass.
    loss = loss_fn(y_pred, y_train)

    optimizer.zero_grad()             # Clear old gradients.
    loss.backward()                   # Backpropagation.
    optimizer.step()                  # Update parameters.

    loss_history.append(loss.item())
    if (epoch + 1) % 500 == 0:
        print(f"Epoch {epoch + 1:4d} | Training MSE: {loss.item():.6f}")


## 4. Inspect the learning curve

A decreasing loss indicates that the predictions are getting closer to the training targets. The logarithmic vertical axis makes small losses easier to compare. Adam's updates may cause some fluctuations; the loss need not decrease at every epoch.

In [ ]:
plt.figure(figsize=(7, 3))
plt.semilogy(range(1, n_epochs + 1), loss_history)
plt.xlabel("Epoch")
plt.ylabel("Training MSE (log scale)")
plt.title("Learning curve")
plt.grid(alpha=0.3)
plt.show()


## 5. Evaluate on new inputs

In [ ]:
x_test = torch.linspace(0, 1, 402)[1:-1].reshape(-1, 1)
y_test = true_function(x_test)

model.eval()
with torch.no_grad():
    y_pred = model(x_test)
    test_mse = loss_fn(y_pred, y_test).item()

print(f"MSE on new inputs: {test_mse:.6f}")

plt.figure(figsize=(8, 4))
plt.plot(x_test.numpy(), y_test.numpy(), label="Exact sine", linewidth=2)
plt.plot(x_test.numpy(), y_pred.numpy(), "--", label="Model", linewidth=2)
plt.scatter(x_train.numpy(), y_train.numpy(), s=20, color="black",
            alpha=0.5, label="Training examples")
plt.xlabel("x")
plt.ylabel("y")
plt.title(r"Model approximates $\sin(2\pi x - 0.2)$")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
n_params = sum(p.numel() for p in model.parameters())
print(f"Number of trainable parameters: {n_params}")